# XGBoost 2.1+ Model — PharmShed Super Dataset

**Author:** Akhila Annireddy  
**Model:** XGBoost 2.1+ multi-class classifier  
**Task:** Multi-class classification — predict which of 217 pharmaceuticals a person is prescribed  
**Features:** Demographics (Age, Sex, Family_income, Insurance_coverage, Race_ethnicity) + Prescription (Quantity, Form, Strength, Day_Supply)  
**Split strategy:** StratifiedGroupKFold (5-fold CV), grouped by Person_ID to prevent data leakage  
**Missing value strategy:**
  - Prescription features (Quantity, Strength, Day_Supply, Form): left as NaN — XGBoost learns the missing direction natively. These NaNs are structurally meaningful: they identify 'no prescriptions' rows and act as a discriminative signal rather than noise.
  - Demographic numeric features (Age, Family_income): median-imputed inside each CV fold (fit on train split only, applied to val split) to prevent leakage.
  - Categorical features (Sex, Insurance_coverage, Race_ethnicity, Form): set to pandas category dtype; XGBoost handles missing categoricals natively.
**Metrics:** Cohen's Kappa, MCC, macro/micro averaged precision, recall, F2 score  

In [4]:
# Install required libraries.
# xgboost 2.1+ has native categorical support and handles missing values via
# learned default directions — no imputation needed for structural NaNs.
# permetrics provides our evaluation metrics.
!pip install 'xgboost>=2.1.0' permetrics

In [5]:
# Load all required libraries.
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, matthews_corrcoef,
    classification_report
)
from permetrics import ClassificationMetric
import warnings
warnings.filterwarnings('ignore')

print("XGBoost version:", xgb.__version__)
# Must be 2.1.0 or higher for native categorical support

XGBoost version: 2.1.4


In [6]:
# Load the super integrated dataset (2014-2021) and metadata.
# The super dataset extends the base integrated_data.csv with prescription
# features: Quantity, Form, Strength, Day_Supply joined on Observation_ID.
# Metadata provides Person_ID which is used only for StratifiedGroupKFold grouping.
super_df = pd.read_csv('../superdataset_construction/super_integrated_data.csv')
metadata = pd.read_csv('../dataset/metadata.csv')

print("Super dataset shape:", super_df.shape)
print("Metadata shape:", metadata.shape)
print("\nSuper dataset columns:", super_df.columns.tolist())
print("Metadata columns:", metadata.columns.tolist())

Super dataset shape: (905728, 11)
Metadata shape: (905728, 6)

Super dataset columns: ['Observation_ID', 'Drug', 'Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity', 'Quantity', 'Form', 'Strength', 'Day_Supply']
Metadata columns: ['Unnamed: 0', 'Observation_ID', 'Person_ID', 'NDC', 'Household_ID', 'Year']


In [7]:
# Drop the auto-generated index columns from both dataframes.
if 'Unnamed: 0' in super_df.columns:
    super_df = super_df.drop(columns=['Unnamed: 0'])
if 'Unnamed: 0' in metadata.columns:
    metadata = metadata.drop(columns=['Unnamed: 0'])

print("Super dataset columns:", super_df.columns.tolist())
print("Shape:", super_df.shape)

Super dataset columns: ['Observation_ID', 'Drug', 'Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity', 'Quantity', 'Form', 'Strength', 'Day_Supply']
Shape: (905728, 11)


In [8]:
# Join Person_ID from metadata onto the super dataset.
# Person_ID is not a model feature — it is used only to group records by person
# in StratifiedGroupKFold so that all rows for the same individual stay
# in the same fold and cannot leak between train and validation splits.
person_id_map = metadata[['Observation_ID', 'Person_ID']]
super_df = super_df.merge(person_id_map, on='Observation_ID', how='left')

print("Columns after join:", super_df.columns.tolist())
print("Shape after join:", super_df.shape)
print("Missing Person_IDs:", super_df['Person_ID'].isnull().sum())

Columns after join: ['Observation_ID', 'Drug', 'Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity', 'Quantity', 'Form', 'Strength', 'Day_Supply', 'Person_ID']
Shape after join: (905728, 12)
Missing Person_IDs: 0


In [9]:
# EDA: confirm unique persons, drugs, distribution, missing values.
print("Unique persons:", super_df['Person_ID'].nunique())
print("Unique drugs:", super_df['Drug'].nunique())

print("\nTop 10 most prescribed drugs:")
print(super_df['Drug'].value_counts().head(10))

print("\nBottom 5 rarest drugs:")
print(super_df['Drug'].value_counts().tail(5))

print("\nMissing values per column:")
print(super_df.isnull().sum())

print("\nMissing value interpretation:")
print("  Quantity / Form / Strength / Day_Supply NaNs correspond to 'no prescriptions' rows.")
print("  These are left as NaN — XGBoost learns the missing direction natively.")
print("  'no prescriptions' count:", (super_df['Drug'] == 'no prescriptions').sum())

Unique persons: 126967
Unique drugs: 217

Top 10 most prescribed drugs:
Drug
no prescriptions    97497
atorvastatin        38557
lisinopril          35851
metformin           33777
amlodipine          28135
metoprolol          25001
albuterol           23188
omeprazole          22770
losartan            18670
gabapentin          18337
Name: count, dtype: int64

Bottom 5 rarest drugs:
Drug
sulfamethoxazole    72
trimethoprim        72
gentamicin          64
piroxicam           61
ivermectin          29
Name: count, dtype: int64

Missing values per column:
Observation_ID            0
Drug                      0
Age                       0
Sex                       0
Family_income             0
Insurance_coverage        0
Race_ethnicity            0
Quantity              97497
Form                  97497
Strength              97497
Day_Supply            97497
Person_ID                 0
dtype: int64

Missing value interpretation:
  Quantity / Form / Strength / Day_Supply NaNs correspond t

## Preprocessing for XGBoost 2.1+ — Super Dataset

**Two-tier missing value strategy:**

1. **Prescription NaNs (Quantity, Strength, Day_Supply, Form)** — left as `NaN`.
   XGBoost's sparsity-aware split-finding algorithm learns an optimal default branch direction for missing values at every tree node. Since all ~97k 'no prescriptions' rows share the same NaN pattern across all prescription columns, XGBoost will quickly learn to route them to the correct leaf. Replacing these NaNs with any value (median, -1, 0) would destroy this structural signal.

2. **Demographic numeric NaNs (Age, Family_income)** — median-imputed **inside the CV loop**.
   Medians are computed on the training fold only and applied to the validation fold. This prevents any leakage from validation data influencing the imputation value. For the final model, medians are computed on the full training set.

3. **Categorical columns (Sex, Insurance_coverage, Race_ethnicity, Form)** — set to `category` dtype.
   XGBoost 2.1+ handles missing categoricals natively via `enable_categorical=True` in DMatrix.

**XGBoost does not require feature scaling** — numeric columns are passed as-is.

In [10]:
# Define columns and fit LabelEncoder once on the full dataset.
# LabelEncoder is fitted before the CV loop so the drug->integer mapping
# is identical across all folds and for the final model.
feature_cols          = ['Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity',
                         'Quantity', 'Form', 'Strength', 'Day_Supply']
categorical_cols      = ['Sex', 'Insurance_coverage', 'Race_ethnicity', 'Form']
numeric_demo_cols     = ['Age', 'Family_income']      # will be median-imputed inside CV loop
numeric_rx_cols       = ['Quantity', 'Strength', 'Day_Supply']  # left as NaN — XGBoost handles natively
target_col            = 'Drug'

le = LabelEncoder()
super_df['Drug_encoded'] = le.fit_transform(super_df[target_col])

print("Unique classes in encoder:", len(le.classes_))
print("Feature columns:", feature_cols)
print("Categorical cols (category dtype + XGBoost native):", categorical_cols)
print("Numeric demo cols (median-imputed inside CV):", numeric_demo_cols)
print("Numeric Rx cols (NaN = no prescription — XGBoost native):", numeric_rx_cols)
print("\nSample drug->integer mapping (first 5):")
for i, drug in enumerate(le.classes_[:5]):
    print(f"  {drug} -> {i}")

Unique classes in encoder: 217
Feature columns: ['Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity', 'Quantity', 'Form', 'Strength', 'Day_Supply']
Categorical cols (category dtype + XGBoost native): ['Sex', 'Insurance_coverage', 'Race_ethnicity', 'Form']
Numeric demo cols (median-imputed inside CV): ['Age', 'Family_income']
Numeric Rx cols (NaN = no prescription — XGBoost native): ['Quantity', 'Strength', 'Day_Supply']

Sample drug->integer mapping (first 5):
  acetaminophen -> 0
  acyclovir -> 1
  adapalene -> 2
  albuterol -> 3
  alendronate -> 4


In [ ]:
# StratifiedGroupKFold 5-fold CV for XGBoost Super Dataset.
# Identical split strategy to base XGBoost and RealMLP so results are comparable.
#
# For each fold:
#   1. Split by Person_ID groups, stratified by Drug
#   2. Shuffle rows so refills are not bunched together
#   3. Compute median on train split for Age and Family_income; apply to val split
#   4. Set categorical dtypes for XGBoost native handling
#   5. Convert to DMatrix with enable_categorical=True
#      (prescription NaNs remain as NaN — XGBoost learns default direction)
#   6. Train XGBoost multi:softmax classifier with early stopping
#   7. Predict and compute all required metrics

sgkf = StratifiedGroupKFold(n_splits=5)

X      = super_df[feature_cols].copy()
y      = super_df['Drug_encoded'].values
groups = super_df['Person_ID'].values

# XGBoost parameters.
# multi:softmax outputs predicted class integers directly.
# tree_method='hist' is fast and memory-efficient for large datasets.
# device='cpu' — change to 'cuda' if NVIDIA GPU available.
# N_ROUNDS=100: early stopping will find the true optimal round per fold.
# early_stopping_rounds=20: stops if val error does not improve for 20 consecutive rounds.
xgb_params = {
    'objective':        'multi:softmax',
    'num_class':        len(le.classes_),
    'eval_metric':      'merror',
    'device':           'cpu',            # change to 'cuda' if GPU available
    'tree_method':      'hist',
    'max_depth':        6,
    'learning_rate':    0.1,
    'subsample':        0.8,
    'colsample_bytree': 1.0,
    'random_state':     42,
    'verbosity':        1,
}
N_ROUNDS       = 100  # max boosting rounds — model will stop earlier if val error plateaus
EARLY_STOPPING = 20   # stop if val error does not improve for 20 consecutive rounds

fold_results          = []
per_drug_recall_folds = []

for fold_num, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups), start=1):
    print(f"\n{'='*50}")
    print(f"FOLD {fold_num}/5")
    print(f"{'='*50}")

    # Split
    X_train_fold = X.iloc[train_idx].copy()
    X_val_fold   = X.iloc[val_idx].copy()
    y_train_fold = y[train_idx]
    y_val_fold   = y[val_idx]

    # Shuffle rows within each fold to prevent ordering artifacts
    rng          = np.random.RandomState(42)
    train_order  = rng.permutation(len(X_train_fold))
    val_order    = rng.permutation(len(X_val_fold))
    X_train_fold = X_train_fold.iloc[train_order].reset_index(drop=True)
    y_train_fold = y_train_fold[train_order]
    X_val_fold   = X_val_fold.iloc[val_order].reset_index(drop=True)
    y_val_fold   = y_val_fold[val_order]

    # Median imputation for demographic numeric columns.
    # Fit on TRAINING fold only to prevent leakage from validation data.
    # Prescription numeric cols (Quantity, Strength, Day_Supply) are NOT imputed —
    # their NaNs encode 'no prescriptions' and are handled natively by XGBoost.
    train_medians = X_train_fold[numeric_demo_cols].median()
    X_train_fold[numeric_demo_cols] = X_train_fold[numeric_demo_cols].fillna(train_medians)
    X_val_fold[numeric_demo_cols]   = X_val_fold[numeric_demo_cols].fillna(train_medians)

    # Set categorical dtypes — XGBoost handles encoding and missing values internally
    for col in categorical_cols:
        X_train_fold[col] = X_train_fold[col].astype('category')
        X_val_fold[col]   = X_val_fold[col].astype('category')

    print(f"Train size: {len(X_train_fold):,} | Val size: {len(X_val_fold):,}")
    print(f"Unique drugs — train: {len(np.unique(y_train_fold))} | val: {len(np.unique(y_val_fold))}")
    print(f"Train medians used for imputation — Age: {train_medians['Age']:.1f}, "
          f"Family_income: {train_medians['Family_income']:.1f}")

    # Convert to DMatrix with enable_categorical=True.
    # Prescription NaNs (Quantity, Strength, Day_Supply) are passed through as NaN.
    # XGBoost will learn the optimal split direction for missing values at each node.
    dtrain = xgb.DMatrix(X_train_fold, label=y_train_fold, enable_categorical=True)
    dval   = xgb.DMatrix(X_val_fold,   label=y_val_fold,   enable_categorical=True)

    # Train
    evals_result = {}
    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=N_ROUNDS,
        evals=[(dtrain, 'train'), (dval, 'val')],
        evals_result=evals_result,
        verbose_eval=10,
        early_stopping_rounds=EARLY_STOPPING,
    )
    best_round = model.best_iteration
    print(f"Fold {fold_num} training complete. Best round: {best_round}")

    # Predict
    y_pred_fold = model.predict(dval).astype(int)

    # Metrics
    acc   = accuracy_score(y_val_fold, y_pred_fold)
    kappa = cohen_kappa_score(y_val_fold, y_pred_fold)
    mcc   = matthews_corrcoef(y_val_fold, y_pred_fold)

    evaluator       = ClassificationMetric(y_val_fold, y_pred_fold)
    macro_precision = evaluator.precision_score(average='macro')
    micro_precision = evaluator.precision_score(average='micro')
    macro_recall    = evaluator.recall_score(average='macro')
    micro_recall    = evaluator.recall_score(average='micro')
    macro_f1        = evaluator.f1_score(average='macro')
    micro_f1        = evaluator.f1_score(average='micro')
    macro_f2        = evaluator.fbeta_score(beta=2, average='macro')
    micro_f2        = evaluator.fbeta_score(beta=2, average='micro')

    fold_results.append({
        'fold':             fold_num,
        'accuracy':         acc,
        'cohen_kappa':      kappa,
        'mcc':              mcc,
        'macro_precision':  macro_precision,
        'micro_precision':  micro_precision,
        'macro_recall':     macro_recall,
        'micro_recall':     micro_recall,
        'macro_f1':         macro_f1,
        'micro_f1':         micro_f1,
        'macro_f2':         macro_f2,
        'micro_f2':         micro_f2,
    })

    # Per-drug recall for ensemble model selection
    report = classification_report(
        y_val_fold, y_pred_fold,
        labels=np.arange(len(le.classes_)),
        target_names=le.classes_,
        output_dict=True,
        zero_division=0
    )
    drug_recalls         = {drug: report[drug]['recall'] for drug in le.classes_ if drug in report}
    drug_recalls['fold'] = fold_num
    per_drug_recall_folds.append(drug_recalls)

    print(f"Fold {fold_num} results:")
    print(f"  Accuracy:     {acc:.4f}")
    print(f"  Cohen Kappa:  {kappa:.4f}")
    print(f"  MCC:          {mcc:.4f}")
    print(f"  Macro Recall: {macro_recall:.4f}")
    print(f"  Micro Recall: {micro_recall:.4f}")
    print(f"  Macro F2:     {macro_f2:.4f}")

print("\n" + "="*50)
print("ALL FOLDS COMPLETE")
print("="*50)

In [ ]:
# Summarize CV results — mean and std per metric across all 5 folds.
# These numbers go directly into the paper.

results_df = pd.DataFrame(fold_results)
print("Per-fold results:")
print(results_df.to_string(index=False))

print("\nMean ± Std across 5 folds:")
metric_cols = [c for c in results_df.columns if c != 'fold']
for col in metric_cols:
    mean = results_df[col].mean()
    std  = results_df[col].std()
    print(f"  {col:25s}: {mean:.4f} ± {std:.4f}")

results_df.to_csv('xgboost_super_cv_results.csv', index=False)
print("\nCV results saved to xgboost_super_cv_results.csv")

In [ ]:
# Average per-drug recall across all 5 folds.
# This CSV goes to Vanessa for ensemble construction.

per_drug_df = pd.DataFrame(per_drug_recall_folds)
drug_cols   = [c for c in per_drug_df.columns if c != 'fold']

mean_drug_recall = per_drug_df[drug_cols].mean().reset_index()
mean_drug_recall.columns = ['Drug', 'Mean_Recall_XGBoost_Super']
mean_drug_recall = mean_drug_recall.sort_values('Mean_Recall_XGBoost_Super', ascending=False)

print("Top 20 drugs by mean recall (super dataset):")
print(mean_drug_recall.head(20).to_string(index=False))

print("\nBottom 20 drugs by mean recall (super dataset):")
print(mean_drug_recall.tail(20).to_string(index=False))

drugs_with_nonzero = (mean_drug_recall['Mean_Recall_XGBoost_Super'] > 0).sum()
print(f"\nDrugs with non-zero mean recall: {drugs_with_nonzero} / {len(mean_drug_recall)}")

mean_drug_recall.to_csv('xgboost_super_per_drug_recall.csv', index=False)
print("\nPer-drug recall saved to xgboost_super_per_drug_recall.csv")

## Final Model — Train on Full Dataset (2014–2021)

After CV confirms performance, train one final model on the full dataset for internal validation on MEPS 2022 and ensemble use.

For the final model, demographic medians are computed on the full training set (2014–2021). Prescription NaNs remain as NaN.

In [ ]:
# Train final XGBoost model on the full super integrated dataset (2014-2021).

X_final = super_df[feature_cols].copy()
y_final = super_df['Drug_encoded'].values

# Compute medians on full training set for use in 2022 validation preprocessing.
# Store as a dict so we can apply identically to 2022 data.
final_train_medians = X_final[numeric_demo_cols].median()
X_final[numeric_demo_cols] = X_final[numeric_demo_cols].fillna(final_train_medians)
print("Final model medians (Age, Family_income):", final_train_medians.to_dict())

for col in categorical_cols:
    X_final[col] = X_final[col].astype('category')

X_final, y_final = shuffle(X_final, y_final, random_state=42)
X_final = X_final.reset_index(drop=True)

print("Final training data size:", X_final.shape)
print("Number of classes:", len(le.classes_))

dtrain_final = xgb.DMatrix(X_final, label=y_final, enable_categorical=True)

print("\nTraining final XGBoost model on super dataset...")
final_model = xgb.train(
    xgb_params,
    dtrain_final,
    num_boost_round=N_ROUNDS,
    verbose_eval=10,
)
print("Final model training complete!")

final_model.save_model('xgboost_super_final_model.ubj')
print("Model saved to xgboost_super_final_model.ubj")

In [ ]:
# Internal validation on MEPS 2022 (held-out test set).
# 2022 is never seen during training or CV.
#
# Preprocessing must mirror training exactly:
#   - Demographic numeric NaNs: filled with final_train_medians (computed on 2014-2021)
#   - Prescription NaNs: left as NaN (same as training — XGBoost handles natively)
#   - Categorical cols: set to category dtype

data_2022 = pd.read_csv('../superdataset_construction/super_data_2022.csv')
if 'Unnamed: 0' in data_2022.columns:
    data_2022 = data_2022.drop(columns=['Unnamed: 0'])

print("2022 super data shape:", data_2022.shape)

# Apply training medians to 2022 demographic columns — do NOT recompute from 2022 data
data_2022[numeric_demo_cols] = data_2022[numeric_demo_cols].fillna(final_train_medians)

# Filter to only drugs the model knows
known_drugs  = set(le.classes_)
unseen_drugs = set(data_2022['Drug'].unique()) - known_drugs
print(f"Unseen drugs in 2022 (will be dropped): {len(unseen_drugs)}")
if unseen_drugs:
    print("Unseen:", unseen_drugs)

data_2022_filtered = data_2022[data_2022['Drug'].isin(known_drugs)].copy()
print(f"2022 rows after filtering: {len(data_2022_filtered):,}")

X_2022 = data_2022_filtered[feature_cols].copy()
for col in categorical_cols:
    X_2022[col] = X_2022[col].astype('category')

y_2022_encoded = le.transform(data_2022_filtered['Drug'])
d2022 = xgb.DMatrix(X_2022, label=y_2022_encoded, enable_categorical=True)

y_pred_2022 = final_model.predict(d2022).astype(int)

# Metrics
acc_2022   = accuracy_score(y_2022_encoded, y_pred_2022)
kappa_2022 = cohen_kappa_score(y_2022_encoded, y_pred_2022)
mcc_2022   = matthews_corrcoef(y_2022_encoded, y_pred_2022)

ev2022            = ClassificationMetric(y_2022_encoded, y_pred_2022)
macro_prec_2022   = ev2022.precision_score(average='macro')
micro_prec_2022   = ev2022.precision_score(average='micro')
macro_recall_2022 = ev2022.recall_score(average='macro')
micro_recall_2022 = ev2022.recall_score(average='micro')
macro_f2_2022     = ev2022.fbeta_score(beta=2, average='macro')
micro_f2_2022     = ev2022.fbeta_score(beta=2, average='micro')

print("INTERNAL VALIDATION — MEPS 2022 Results (Super Dataset)")
print(f"Accuracy:          {acc_2022:.4f}")
print(f"Cohen Kappa:       {kappa_2022:.4f}")
print(f"MCC:               {mcc_2022:.4f}")
print(f"Macro Precision:   {macro_prec_2022:.4f}")
print(f"Micro Precision:   {micro_prec_2022:.4f}")
print(f"Macro Recall:      {macro_recall_2022:.4f}")
print(f"Micro Recall:      {micro_recall_2022:.4f}")
print(f"Macro F2:          {macro_f2_2022:.4f}")
print(f"Micro F2:          {micro_f2_2022:.4f}")

# Per-drug metrics on 2022
report_2022 = classification_report(
    y_2022_encoded, y_pred_2022,
    labels=np.arange(len(le.classes_)),
    target_names=le.classes_,
    output_dict=True,
    zero_division=0
)
drug_metrics_2022 = pd.DataFrame([
    {'Drug': drug,
     'Recall_2022':    report_2022[drug]['recall'],
     'Precision_2022': report_2022[drug]['precision'],
     'F1_2022':        report_2022[drug]['f1-score'],
     'Support_2022':   report_2022[drug]['support']}
    for drug in le.classes_ if drug in report_2022
]).sort_values('Recall_2022', ascending=False)

print("\nTop 15 drugs by recall on 2022:")
print(drug_metrics_2022.head(15).to_string(index=False))
print("\nBottom 15 drugs by recall on 2022:")
print(drug_metrics_2022.tail(15).to_string(index=False))

drug_metrics_2022.to_csv('xgboost_super_2022_per_drug_metrics.csv', index=False)

pd.DataFrame([{
    'model':            'XGBoost_Super',
    'dataset':          'MEPS_2022_internal_validation',
    'accuracy':         acc_2022,
    'cohen_kappa':      kappa_2022,
    'mcc':              mcc_2022,
    'macro_precision':  macro_prec_2022,
    'micro_precision':  micro_prec_2022,
    'macro_recall':     macro_recall_2022,
    'micro_recall':     micro_recall_2022,
    'macro_f2':         macro_f2_2022,
    'micro_f2':         micro_f2_2022,
}]).to_csv('xgboost_super_validation_summary.csv', index=False)
print("\nAll 2022 results saved.")

In [ ]:
# Inspect prediction distribution on 2022 data.
# ratio > 1 = over-predicted, ratio < 1 = under-predicted, 0 = never predicted.
# Compare with base XGBoost to assess whether prescription features improve rare drug recall.

pred_drugs_2022   = le.inverse_transform(y_pred_2022)
actual_drugs_2022 = le.inverse_transform(y_2022_encoded)

pred_counts   = pd.Series(pred_drugs_2022).value_counts().rename('predicted')
actual_counts = pd.Series(actual_drugs_2022).value_counts().rename('actual')

dist_compare = pd.concat([actual_counts, pred_counts], axis=1).fillna(0).astype(int)
dist_compare['ratio_pred_to_actual'] = (
    dist_compare['predicted'] / dist_compare['actual'].replace(0, 1)
).round(2)
dist_compare = dist_compare.sort_values('actual', ascending=False)

print("Prediction vs actual (top 20 most common drugs):")
print(dist_compare.head(20))
print("\nPrediction vs actual (bottom 20 rarest drugs):")
print(dist_compare.tail(20))

never_predicted = dist_compare[dist_compare['predicted'] == 0]
print(f"\nDrugs never predicted: {len(never_predicted)}")
if len(never_predicted) > 0:
    print(never_predicted.index.tolist())

In [ ]:
# Feature importance from XGBoost (by gain).
# With prescription features now included, we expect Quantity, Strength, Day_Supply
# to rank highly — they are drug-specific and directly discriminative.
# Compare the ranking shift vs base XGBoost (demographics only) for the paper discussion.

importance = final_model.get_score(importance_type='gain')
importance_df = pd.DataFrame([
    {'Feature': k, 'Importance_Gain': v}
    for k, v in importance.items()
]).sort_values('Importance_Gain', ascending=False)

print("Feature importance by gain (super dataset — demographics + prescription):")
print(importance_df.to_string(index=False))

importance_df.to_csv('xgboost_super_feature_importance.csv', index=False)
print("\nFeature importance saved to xgboost_super_feature_importance.csv")

## Summary of Outputs

| File | Contents |
|------|----------|
| `xgboost_super_cv_results.csv` | Mean ± std for all metrics across 5 CV folds |
| `xgboost_super_per_drug_recall.csv` | Average recall per drug across 5 folds (for ensemble) |
| `xgboost_super_2022_per_drug_metrics.csv` | Per-drug recall, precision, F1 on MEPS 2022 |
| `xgboost_super_validation_summary.csv` | Overall validation metrics on MEPS 2022 |
| `xgboost_super_feature_importance.csv` | Feature importance by gain (demographics + prescription features) |
| `xgboost_super_final_model.ubj` | Saved final model |

**Key difference from base XGBoost:** Prescription features (Quantity, Form, Strength, Day_Supply) are added. Their NaNs are left intact for XGBoost native handling — they structurally identify 'no prescriptions' rows. Demographic numeric NaNs (Age, Family_income) are median-imputed fold-by-fold to prevent leakage.